# Goal

Try the workflow in "simplest_neural_model" out on sequences we've extracted.


## Expected raw format

Each example should be a pair `(sequence, label)` where:

- `sequence` is a list of string tokens, such as `["A", "dog", "hel", "o"]`
- `label` is a class id, string, or any hashable value; one-element lists like `[5]` also work

Examples:
```python
raw_pairs = [
    (["A", "C", "E"], 1),
    (["Q", "dog", "seven"], 5),
    (["A", "E"], 1),
    (["Q", "seven"], 5),
]
```

Below, the notebook creates a **toy synthetic dataset** by default so the notebook runs as-is.  
For real use, replace the synthetic `raw_pairs` cell with your own data.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "path_matcher").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/path_matcher.")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
for path in (str(REPO_ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Repo root:", REPO_ROOT)


In [ ]:
import sys
import os.path

In [ ]:
from pathlib import Path
import random
import torch

from exemplar.neural.sequence_class_utils import (
    PAD_TOKEN,
    BOS_TOKEN,
    EOS_TOKEN,
    UNK_TOKEN,
    decode_token_ids,
    encode_raw_pairs,
    group_encoded_pairs_by_label,
    infer_max_raw_length,
    summarize_encoded_dataset,
    average_sequence_by_length,
)
from exemplar.neural.set_conditioned_generator import (
    ModelConfig,
    TrainingConfig,
    generate_representatives,
    load_checkpoint,
    predict_length_distribution,
    save_checkpoint,
    train_model,
)

torch.set_num_threads(1)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CPU threads:", torch.get_num_threads())


## Generate toy sequence data

This benchmark uses a small synthetic sequence dataset rather than a committed pickle file.


In [ ]:
def apply_indel_noise(template, extra_vocab, rng, delete_p=0.15, insert_p=0.15):
    out = []
    for tok in template:
        if rng.random() > delete_p:
            out.append(tok)
        if rng.random() < insert_p:
            out.append(rng.choice(extra_vocab))
    if rng.random() < insert_p:
        pos = rng.randrange(len(out) + 1)
        out.insert(pos, rng.choice(extra_vocab))
    if len(out) == 0:
        out = [rng.choice(template)]
    return out


def make_toy_dataset(seed=0, n_per_class=80):
    rng = random.Random(seed)
    templates = {
        0: ["A", "B", "C", "D", "E"],
        1: ["dog", "runs", "to", "park"],
        2: ["red", "green", "blue", "yellow"],
        3: ["alpha", "beta", "gamma"],
    }
    extra_vocab = ["x", "y", "z", "fast", "slow", "north", "south", "cat", "seven", "moon"]
    raw = []
    for label, template in templates.items():
        for _ in range(n_per_class):
            raw.append((apply_indel_noise(template, extra_vocab, rng, delete_p=0.18, insert_p=0.18), label))
    rng.shuffle(raw)
    return raw, templates


raw_pairs, true_templates = make_toy_dataset(seed=0, n_per_class=80)


In [ ]:
len(raw_pairs), true_templates


## Neural benchmark workflow


## Encode the data

This learns a token vocabulary and converts string sequences to integer sequences.  
The class labels are **not** embedded into the model; they are only used to form class-specific support sets during training.


In [ ]:
encoded_pairs, token_to_id, id_to_token = encode_raw_pairs(raw_pairs, min_freq=1)
grouped_sequences = group_encoded_pairs_by_label(encoded_pairs)
summary = summarize_encoded_dataset(encoded_pairs)

print("Dataset summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

print("\nVocabulary size:", len(token_to_id))
print("Special token ids:")
for tok in [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]:
    print(f"  {tok}: {token_to_id[tok]}")


## Set model and training hyperparameters

A good default is to set `max_seq_len = max_raw_length + 2`, which leaves room for BOS/EOS.


In [ ]:
max_raw_len = infer_max_raw_length(encoded_pairs)

model_config = ModelConfig(
    vocab_size=len(token_to_id),
    pad_id=token_to_id[PAD_TOKEN],
    bos_id=token_to_id[BOS_TOKEN],
    eos_id=token_to_id[EOS_TOKEN],
    max_seq_len=max_raw_len + 2,
    d_model=128,
    num_seq_layers=1,
    num_dec_layers=1,
    dropout=0.1,
)

training_config = TrainingConfig(
    min_support_size=2,
    max_support_size=6,
    batch_size=32,
    epochs=50,
    episodes_per_epoch=250,
    lr=3e-4,
    weight_decay=1e-4,
    lambda_len=0.7,
    grad_clip=1.0,
    seed=7,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_workers=0,
    verbose=True,
    num_torch_threads=1,
)

model_config, training_config


## Train the model
For real data, you will usually want more epochs and often a larger `episodes_per_epoch`.


In [ ]:
model, history = train_model(
    grouped_sequences=grouped_sequences,
    model_config=model_config,
    training_config=training_config,
)

print("\nTraining history:")
for row in history:
    print(row)


## Save the checkpoint outside version-controlled source
The checkpoint contains:

- the model weights
- the model and training configs
- the token vocabulary
- the training history


In [ ]:
checkpoint_path = REPO_ROOT / ".cache" / "checkpoints" / "set_conditioned_generator_demo.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

save_checkpoint(
    path=str(checkpoint_path),
    model=model,
    model_config=model_config,
    training_config=training_config,
    token_to_id=token_to_id,
    id_to_token=id_to_token,
    history=history,
    extra_metadata={"note": "Demo checkpoint from the notebook."},
)

print("Saved checkpoint to:", checkpoint_path.resolve())


## Reload the saved model
This is optional, but demonstrates the intended workflow.


In [ ]:
artifact = load_checkpoint(str(checkpoint_path), map_location=training_config.device)
reloaded_model = artifact["model"]
print("Reloaded model on device:", next(reloaded_model.parameters()).device)


## Generate representative sequences for one class

At inference time, you can condition on **all** sequences from a class, or on a subset of them.
The helper below uses all sequences from the chosen class.


In [ ]:
chosen_label = sorted(grouped_sequences.keys())[0]
support_sequences = grouped_sequences[chosen_label]

print("Chosen label:", chosen_label)
print("Number of support sequences used at inference:", len(support_sequences))

length_probs = predict_length_distribution(reloaded_model, support_sequences)
top_lengths = torch.topk(length_probs[1:], k=min(6, len(length_probs) - 1)).indices + 1

print("\nTop predicted lengths:")
for L in top_lengths.tolist():
    print(f"  length={L:2d}  prob={float(length_probs[L]):.4f}")

generated = generate_representatives(reloaded_model, support_sequences, n_outputs=5)

print("\nGenerated representatives:")
for i, item in enumerate(generated, start=1):
    decoded = decode_token_ids(item["token_ids"], id_to_token=id_to_token)
    print(
        f"  {i}. predicted_length={item['predicted_length']} "
        f"(prob={item['length_probability']:.4f}) -> {decoded}"
    )


## Compare against a few real sequences from that class
On the toy dataset, these should often resemble the hidden class template.


In [ ]:
print("Some real sequences from the chosen class:")
for seq in support_sequences[:8]:
    print(" ", decode_token_ids(seq, id_to_token=id_to_token))

if "true_templates" in globals() and chosen_label in true_templates:
    print("\nToy hidden template for this class:")
    print(" ", true_templates[chosen_label])


In [ ]:
print("Averaged sequences across many runs:")

many_generated = generate_representatives(reloaded_model, support_sequences, n_outputs=2000)
many_decoded = [decode_token_ids(item["token_ids"], id_to_token=id_to_token) for item in many_generated]

In [ ]:
avg = average_sequence_by_length(
    many_decoded,
)
# print(avg[5]["position_token_probs"][2])

In [ ]:
import numpy as np 

gt = true_templates[chosen_label]
est = avg[len(true_templates[chosen_label])]["average_sequence"]

print("gt:" + str(gt))
print("est:" + str(est))
print("score:" + str(np.sum([gt[i] == est[i] for i in range(len(gt))])/len(gt)))